# 학생 성취도 예측 프로젝트 — EDA

**데이터**: `hybrid_student_performance_1200.csv` (1,200행 × 36열)  
**분석 목표**  
- 타겟 ① `cgpa_category` — CGPA 구간 예측 (회귀 대용 분류)  
- 타겟 ② `performance_risk_level` — 위험도 분류 (저 / 중 / 고위험)  
- 핵심 질문: 공부 시간·출석률·스트레스·수면 등 행동 변수가 성적에 얼마나 영향을 미치는가?  

**학번**: 20242530 정명진

---
## 0. 환경 설정 및 번역 사전

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# ── 컬럼명 한국어 ──────────────────────────────────────────────────
COL_KO = {
    'age': '나이', 'gender': '성별', 'year_class': '학년',
    'program_stream': '전공 트랙', 'cgpa_category': 'CGPA 구간',
    'academic_satisfaction': '학업 만족도', 'study_hours_daily': '일일 공부 시간',
    'daily_productivity': '일일 생산성', 'revision_frequency': '복습 빈도',
    'focus_duration': '집중 지속 시간', 'screen_time_non_study': '비학습 스크린 타임',
    'main_distractor': '주요 방해 요소', 'study_consistency': '공부 일관성',
    'tasks_on_time': '과제 제때 제출', 'preparation_status': '시험 준비 상태',
    'career_goal_clarity': '진로 목표 명확도', 'skills_developing': '개발 역량 유형',
    'energy_level': '에너지 레벨', 'stress_level': '스트레스 레벨',
    'routine_rating': '루틴 평가', 'sleepy_during_study': '공부 중 졸림',
    'sleep_hours': '수면 시간', 'career_interest': '진로 관심사',
    'online_courses': '온라인 강좌', 'projects_internships': '프로젝트/인턴십',
    'programming_foundation': '프로그래밍 기초', 'events_participation': '행사 참여',
    'assignments_on_time': '과제 제출', 'attendance_percentage': '출석률',
    'strongest_asset': '주요 강점', 'internal_barrier': '내부 장벽',
    'external_resources': '외부 자료 활용', 'external_pressure': '외부 압박',
    'performance_risk_level': '성과 위험도',
}

# ── 값 한국어 ─────────────────────────────────────────────────────
VAL_KO = {
    'gender': {'Female': '여성', 'Male': '남성', 'Other': '기타'},
    'year_class': {
        'First Year (FY)': '1학년', 'Second Year (SY)': '2학년',
        'Third Year (TY)': '3학년', 'Final Year': '4학년',
        'First Year (PG)': '대학원 1학년',
    },
    'program_stream': {
        'BCA': 'BCA', 'BCom': 'BCom', 'BSc Cyber Security': '사이버보안',
        'BSc IT': 'IT학과', 'BA': '인문학과', 'BSc CS': '컴퓨터과학', 'BBA': '경영학과',
    },
    'cgpa_category': {
        '5.0 \u2013 6.9': '5.0~6.9', '7.0 \u2013 8.4': '7.0~8.4',
        '8.5 \u2013 9.4': '8.5~9.4', '9.5 \u2013 10.0': '9.5~10.0',
    },
    'academic_satisfaction': {
        'Very unsatisfied': '매우 불만족', 'Unsatisfied': '불만족',
        'Neutral': '보통', 'Satisfied': '만족', 'Very satisfied': '매우 만족',
    },
    'study_hours_daily': {
        'Less than 1 hour': '1시간 미만', '1\u20132 hours': '1~2시간',
        'More than 2 hours': '2시간 초과',
    },
    'revision_frequency': {
        'Never': '안 함', 'Rarely': '거의 안 함',
        'Few times a week': '주 몇 번', 'Daily': '매일',
    },
    'focus_duration': {
        '30\u201360 minutes': '30~60분', '1\u20132 hours': '1~2시간',
        'More than 2 hours': '2시간 초과',
    },
    'screen_time_non_study': {
        '2\u20134 hours': '2~4시간', '4\u20136 hours': '4~6시간',
        'More than 6 hours': '6시간 초과',
    },
    'main_distractor': {
        'Social media': 'SNS', 'Video content (YouTube/OTT)': '유튜브/OTT',
        'Social interactions': '친구/사교', 'Gaming': '게임', 'Other': '기타',
    },
    'study_consistency': {
        'Rarely': '거의 안 함', 'Sometimes': '가끔', 'Mostly consistent': '대체로 일정',
    },
    'tasks_on_time': {
        'Rarely': '거의 안 함', 'Sometimes': '가끔', 'Often': '자주', 'Always': '항상',
    },
    'preparation_status': {
        'Planning to start soon': '곧 시작 예정',
        'Actively preparing for a goal (placements/exams)': '적극 준비 중',
        'Thinking about it': '고민 중',
    },
    'career_goal_clarity': {
        'Not clear': '불명확', 'Somewhat clear': '어느 정도 명확', 'Very clear': '매우 명확',
    },
    'skills_developing': {
        'Hard skills (programming, data analytics, technical skills)': '하드 스킬',
        'Both hard and soft skills': '하드+소프트 스킬',
        'Soft skills (communication, teamwork, leadership, financial literacy)': '소프트 스킬',
    },
    'sleepy_during_study': {
        'Never': '안 졸림', 'Sometimes': '가끔', 'Often': '자주', 'Always': '항상',
    },
    'sleep_hours': {
        '4\u20135 hours': '4~5시간', '6\u20137 hours': '6~7시간',
        'More than 8 hours': '8시간 초과',
    },
    'career_interest': {
        'Other': '기타', 'Automation Engineer': '자동화 엔지니어',
        'Cyber Security Analyst': '사이버보안 분석가', 'Data Analyst': '데이터 분석가',
        'AI / ML': 'AI/ML', 'Software Developer': '소프트웨어 개발자',
        'Web Developer': '웹 개발자',
    },
    'online_courses': {
        'Not currently, but intend to in the future': '미래 계획 있음',
        'Yes, currently enrolled in one or more courses/certifications': '현재 수강 중',
        'Planning to enroll soon': '곧 등록 예정',
        'No, not interested': '관심 없음',
    },
    'projects_internships': {
        'Yes, actively working on projects/internship': '적극 참여 중',
        'Planning to start a project/internship soon': '곧 시작 예정',
        'Not currently, but intend to in the future': '미래 계획 있음',
    },
    'programming_foundation': {
        'Limited knowledge, theoretical only': '이론적 지식만',
        'Basic knowledge, learning while practicing': '기본 · 실습 중',
        'Strong foundation in core concepts': '핵심 개념 탄탄',
    },
    'events_participation': {
        'Never participate in such events': '참여 안 함',
        'Rarely participate, mostly observe': '관찰 위주',
        'Occasionally participate in events': '가끔 참여',
    },
    'assignments_on_time': {
        'Rarely': '거의 안 함', 'Sometimes': '가끔', 'Often': '자주', 'Always': '항상',
    },
    'attendance_percentage': {
        'Less than 50%': '50% 미만', '50% \u2013 65%': '50~65%',
        '66% \u2013 75%': '66~75%', '76% \u2013 85%': '76~85%',
        'Above 85%': '85% 초과',
    },
    'strongest_asset': {
        'Technical/Hard Skills (Coding, Math, Logic)': '기술적 역량',
        'Creative/Design Skills (Innovation, UI/UX, Content)': '창의/디자인',
        'Management/Execution (Planning, Organizing, Discipline)': '관리/실행력',
        'Soft Skills (Communication, Leadership, Teamwork)': '소프트 스킬',
    },
    'internal_barrier': {
        'Lack of Consistency or Determination (Difficulty sticking to a plan)': '의지/지속성 부족',
        'Difficulty with Focus / Concentration': '집중력 부족',
        'Procrastination / Low Motivation': '미루는 습관',
        'Poor Time Management / Over-scheduling': '시간 관리 미흡',
    },
    'external_resources': {
        'Never (Unaware or Not interested)': '전혀 안 함',
        'Rarely (Passive)': '거의 안 함', 'Occasionally (When needed)': '가끔',
    },
    'external_pressure': {
        'No Impact (Fully supportive environment)': '영향 없음',
        'Low Impact (Rarely affects study)': '낮은 영향',
        'Moderate Impact (Occasional disruption)': '보통 영향',
        'High Impact (Frequent disruption)': '높은 영향',
    },
    'performance_risk_level': {
        'Low Risk': '저위험', 'Moderate Risk': '중위험', 'High Risk': '고위험',
    },
}

def ko(col):
    return COL_KO.get(col, col)

def tdf(df, cols):
    """지정 컬럼을 한국어로 번역한 복사본 반환."""
    d = df.copy()
    for c in cols:
        if c in VAL_KO:
            d[c] = d[c].map(lambda x: VAL_KO[c].get(str(x), str(x)))
    return d

# ── 한국어 순서 리스트 ─────────────────────────────────────────────
CGPA_ORDER_EN  = ['5.0 \u2013 6.9', '7.0 \u2013 8.4', '8.5 \u2013 9.4', '9.5 \u2013 10.0']
RISK_ORDER_EN  = ['Low Risk', 'Moderate Risk', 'High Risk']
STUDY_ORDER_EN = ['Less than 1 hour', '1\u20132 hours', 'More than 2 hours']
SLEEP_ORDER_EN = ['4\u20135 hours', '6\u20137 hours', 'More than 8 hours']
ATT_ORDER_EN   = ['Less than 50%', '50% \u2013 65%', '66% \u2013 75%', '76% \u2013 85%', 'Above 85%']
SAT_ORDER_EN   = ['Very unsatisfied', 'Unsatisfied', 'Neutral', 'Satisfied', 'Very satisfied']

CGPA_ORDER_KO  = [VAL_KO['cgpa_category'][v]         for v in CGPA_ORDER_EN]
RISK_ORDER_KO  = [VAL_KO['performance_risk_level'][v] for v in RISK_ORDER_EN]
STUDY_ORDER_KO = [VAL_KO['study_hours_daily'][v]      for v in STUDY_ORDER_EN]
SLEEP_ORDER_KO = [VAL_KO['sleep_hours'][v]            for v in SLEEP_ORDER_EN]
ATT_ORDER_KO   = [VAL_KO['attendance_percentage'][v]  for v in ATT_ORDER_EN]
SAT_ORDER_KO   = [VAL_KO['academic_satisfaction'][v]  for v in SAT_ORDER_EN]

RISK_COLOR_KO = {'저위험': '#4CAF50', '중위험': '#FFC107', '고위험': '#F44336'}

print('설정 완료')

---
## 1. 데이터 로드 및 기본 정보

In [ ]:
DATA_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                         'archive', 'hybrid_student_performance_1200.csv')
df = pd.read_csv(DATA_PATH)
print(f'데이터 형태: {df.shape[0]}행 × {df.shape[1]}열')

# 실제 존재하는 값만 추린 한국어 순서
cgpa_ko  = [VAL_KO['cgpa_category'][v]         for v in CGPA_ORDER_EN  if v in df['cgpa_category'].values]
risk_ko  = [VAL_KO['performance_risk_level'][v] for v in RISK_ORDER_EN  if v in df['performance_risk_level'].values]
study_ko = [VAL_KO['study_hours_daily'][v]      for v in STUDY_ORDER_EN if v in df['study_hours_daily'].values]
sleep_ko = [VAL_KO['sleep_hours'][v]            for v in SLEEP_ORDER_EN if v in df['sleep_hours'].dropna().values]
att_ko   = [VAL_KO['attendance_percentage'][v]  for v in ATT_ORDER_EN   if v in df['attendance_percentage'].values]
sat_ko   = [VAL_KO['academic_satisfaction'][v]  for v in SAT_ORDER_EN   if v in df['academic_satisfaction'].values]

df.head(3)

In [ ]:
# 컬럼 정보 (한국어 컬럼명 포함)
info = pd.DataFrame({
    '한국어 컬럼명': [COL_KO.get(c, c) for c in df.columns],
    '타입':          df.dtypes.values.astype(str),
    '결측치':        df.isnull().sum().values,
    '결측률(%)':     (df.isnull().sum().values / len(df) * 100).round(2),
    '유니크 수':     df.nunique().values,
}, index=df.columns)
display(info)

In [ ]:
# 수치형 기술통계 (한국어 인덱스)
numeric_cols = df.select_dtypes(include='number').columns.tolist()
desc = df[numeric_cols].describe().T
desc.index = [ko(c) for c in desc.index]
desc.round(3).style.background_gradient(cmap='Blues', subset=['mean', 'std'])

In [ ]:
print(f'중복 행: {df.duplicated().sum()}개')

# 타겟 분포 (한국어 출력)
df_t = tdf(df, ['cgpa_category', 'performance_risk_level'])
print('\n[타겟 ①] CGPA 구간')
print(df_t['cgpa_category'].value_counts().reindex(cgpa_ko, fill_value=0))
print('\n[타겟 ②] 성과 위험도')
print(df_t['performance_risk_level'].value_counts().reindex(risk_ko, fill_value=0))

---
## 2. 타겟 변수 분포

In [ ]:
df_t = tdf(df, ['cgpa_category', 'performance_risk_level'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ① CGPA 구간 막대
cgpa_cnt = df_t['cgpa_category'].value_counts().reindex(cgpa_ko, fill_value=0)
bars = axes[0].bar(cgpa_cnt.index, cgpa_cnt.values,
                   color=sns.color_palette('Blues_d', len(cgpa_cnt)), edgecolor='white')
for b in bars:
    axes[0].text(b.get_x() + b.get_width() / 2, b.get_height() + 3,
                 int(b.get_height()), ha='center', fontsize=9)
axes[0].set_title('CGPA 구간 분포 (회귀 타겟)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('CGPA 구간')
axes[0].set_ylabel('학생 수')
axes[0].tick_params(axis='x', rotation=15)

# ② 위험도 파이
risk_cnt = df_t['performance_risk_level'].value_counts().reindex(risk_ko, fill_value=0)
axes[1].pie(risk_cnt.values, labels=risk_cnt.index, autopct='%1.1f%%',
            colors=[RISK_COLOR_KO[r] for r in risk_ko], startangle=140,
            textprops={'fontsize': 11}, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('성과 위험도 분포 (분류 타겟)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 3. 수치형 변수 분포

In [ ]:
NUM_FEATS = ['age', 'daily_productivity', 'energy_level', 'stress_level', 'routine_rating']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()
palette = sns.color_palette('muted', len(NUM_FEATS))

for i, col in enumerate(NUM_FEATS):
    data = df[col].dropna()
    axes[i].hist(data, bins=20, color=palette[i], edgecolor='white', alpha=0.85)
    axes[i].axvline(data.mean(),   color='red',    linestyle='--', lw=1.5,
                    label=f'평균 {data.mean():.2f}')
    axes[i].axvline(data.median(), color='orange', linestyle=':',  lw=1.5,
                    label=f'중앙값 {data.median():.2f}')
    axes[i].set_title(ko(col), fontsize=11, fontweight='bold')
    axes[i].set_xlabel('값')
    axes[i].set_ylabel('빈도')
    axes[i].legend(fontsize=8)

axes[-1].axis('off')
plt.suptitle('수치형 변수 히스토그램', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 위험도별 박스플롯
df_b = tdf(df, ['performance_risk_level'])

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(NUM_FEATS):
    sns.boxplot(data=df_b, x='performance_risk_level', y=col,
                order=risk_ko, palette=RISK_COLOR_KO,
                ax=axes[i], width=0.5, flierprops={'marker': 'o', 'markersize': 3})
    axes[i].set_title(f'{ko(col)} by 위험도', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('성과 위험도')
    axes[i].set_ylabel(ko(col))
    axes[i].tick_params(axis='x', rotation=15)

axes[-1].axis('off')
plt.suptitle('위험도별 수치형 변수 (박스플롯)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 4. 범주형 변수 분포

In [ ]:
cat_plots = [
    ('gender',                None,           '성별'),
    ('year_class',            None,           '학년'),
    ('program_stream',        None,           '전공 트랙'),
    ('study_hours_daily',     STUDY_ORDER_EN, '일일 공부 시간'),
    ('sleep_hours',           SLEEP_ORDER_EN, '수면 시간'),
    ('attendance_percentage', ATT_ORDER_EN,   '출석률'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
pal = sns.color_palette('pastel', 8)

for i, (col, order_en, title) in enumerate(cat_plots):
    df_c = tdf(df, [col])
    if order_en:
        order_ko = [VAL_KO[col][v] for v in order_en if v in df[col].unique()]
    else:
        order_ko = df_c[col].value_counts().index.tolist()
    counts = df_c[col].value_counts().reindex(order_ko, fill_value=0)
    bars = axes[i].barh(counts.index[::-1], counts.values[::-1],
                        color=pal[:len(counts)], edgecolor='white')
    for b in bars:
        axes[i].text(b.get_width() + 1, b.get_y() + b.get_height() / 2,
                     int(b.get_width()), va='center', fontsize=8)
    axes[i].set_title(title, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('학생 수')
    axes[i].set_xlim(0, counts.max() * 1.14)

plt.suptitle('주요 범주형 변수 분포', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 방해 요소 / 내부 장벽 / 주요 강점
bar_plots = [
    ('main_distractor',  '주요 방해 요소'),
    ('internal_barrier', '내부 장벽'),
    ('strongest_asset',  '주요 강점'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (col, title) in zip(axes, bar_plots):
    df_c = tdf(df, [col])
    counts = df_c[col].value_counts()
    bars = ax.barh(counts.index[::-1], counts.values[::-1],
                   color=sns.color_palette('Set2', len(counts)), edgecolor='white')
    for b in bars:
        ax.text(b.get_width() + 1, b.get_y() + b.get_height() / 2,
                int(b.get_width()), va='center', fontsize=8)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('학생 수')

plt.tight_layout()
plt.show()

---
## 5. 수치형 변수 상관관계 히트맵

In [ ]:
corr = df[numeric_cols].corr()
corr.columns = [ko(c) for c in corr.columns]
corr.index   = [ko(c) for c in corr.index]

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            linewidths=0.5, annot_kws={'size': 10}, ax=ax)
ax.set_title('수치형 변수 상관관계 히트맵', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

# 상관계수 절댓값 상위 5쌍
corr_pairs = (corr.where(~mask)
                  .stack().reset_index()
                  .rename(columns={'level_0': '변수1', 'level_1': '변수2', 0: '상관계수'}))
print('\n▶ 상관계수 절댓값 상위 5쌍')
print(corr_pairs.assign(abs_r=corr_pairs['상관계수'].abs())
                .sort_values('abs_r', ascending=False)
                .drop('abs_r', axis=1).head(5).to_string(index=False))

---
## 6. 주요 변수 × 타겟 관계 분석 (심화)

In [ ]:
# 6-1. 출석률 × CGPA 구간 교차 히트맵
df_c = tdf(df, ['attendance_percentage', 'cgpa_category'])
cross = (pd.crosstab(df_c['attendance_percentage'], df_c['cgpa_category'])
           .reindex(att_ko, fill_value=0)
           .reindex(columns=cgpa_ko, fill_value=0))

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(cross, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('출석률 × CGPA 구간 교차 빈도', fontsize=13, fontweight='bold')
ax.set_xlabel('CGPA 구간')
ax.set_ylabel('출석률')
plt.tight_layout()
plt.show()

In [ ]:
# 6-2. 일일 공부 시간 × CGPA — 빈도 + 누적 비율 막대
df_c = tdf(df, ['study_hours_daily', 'cgpa_category'])
cross = (pd.crosstab(df_c['study_hours_daily'], df_c['cgpa_category'])
           .reindex(study_ko, fill_value=0)
           .reindex(columns=cgpa_ko, fill_value=0))
cross_pct = cross.div(cross.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
cross.plot(kind='bar', color=sns.color_palette('Blues', len(cgpa_ko)),
           ax=axes[0], edgecolor='white')
axes[0].set_title('공부 시간 × CGPA 구간 (빈도)', fontweight='bold')
axes[0].set_xlabel('일일 공부 시간')
axes[0].set_ylabel('학생 수')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend(title='CGPA 구간', fontsize=8, bbox_to_anchor=(1, 1))

cross_pct.plot(kind='bar', stacked=True,
               color=sns.color_palette('Blues', len(cgpa_ko)),
               ax=axes[1], edgecolor='white')
axes[1].set_title('공부 시간 × CGPA 구간 (누적 비율)', fontweight='bold')
axes[1].set_xlabel('일일 공부 시간')
axes[1].set_ylabel('비율 (%)')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(title='CGPA 구간', fontsize=8, bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

In [ ]:
# 6-3. 스트레스 vs 에너지 산점도 (위험도별)
df_c = tdf(df, ['performance_risk_level'])

fig, ax = plt.subplots(figsize=(8, 6))
for risk in risk_ko:
    g = df_c[df_c['performance_risk_level'] == risk]
    ax.scatter(g['stress_level'], g['energy_level'],
               c=RISK_COLOR_KO[risk], label=risk,
               alpha=0.5, s=35, edgecolors='white', linewidths=0.3)
ax.set_title('스트레스 vs 에너지 레벨 (위험도별)', fontsize=13, fontweight='bold')
ax.set_xlabel('스트레스 레벨')
ax.set_ylabel('에너지 레벨')
ax.legend(title='성과 위험도')
plt.tight_layout()
plt.show()

In [ ]:
# 6-4. 위험도별 KDE (스트레스 / 에너지)
df_c = tdf(df, ['performance_risk_level'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for col, ax in zip(['stress_level', 'energy_level'], axes):
    for risk in risk_ko:
        g = df_c[df_c['performance_risk_level'] == risk][col].dropna()
        sns.kdeplot(g, ax=ax, label=risk, color=RISK_COLOR_KO[risk], fill=True, alpha=0.25)
    ax.set_title(f'{ko(col)} 밀도 분포 (위험도별)', fontsize=11, fontweight='bold')
    ax.set_xlabel(ko(col))
    ax.set_ylabel('밀도')
    ax.legend(title='성과 위험도')
plt.tight_layout()
plt.show()

In [ ]:
# 6-5. 수면 시간 × 위험도 카운트플롯
df_c = tdf(df, ['sleep_hours', 'performance_risk_level'])

fig, ax = plt.subplots(figsize=(11, 5))
sns.countplot(data=df_c, x='sleep_hours', hue='performance_risk_level',
              order=sleep_ko, hue_order=risk_ko,
              palette=RISK_COLOR_KO, ax=ax, edgecolor='white')
ax.set_title('수면 시간별 위험도 분포', fontsize=13, fontweight='bold')
ax.set_xlabel('수면 시간')
ax.set_ylabel('학생 수')
ax.legend(title='성과 위험도')
plt.tight_layout()
plt.show()

In [ ]:
# 6-6. CGPA별 생산성 / 스트레스 바이올린
df_c = tdf(df, ['cgpa_category'])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, col, pal in zip(axes, ['daily_productivity', 'stress_level'], ['Blues', 'Reds']):
    sns.violinplot(data=df_c, x='cgpa_category', y=col,
                   order=cgpa_ko, palette=pal, ax=ax, cut=0, inner='quartile')
    ax.set_title(f'CGPA 구간별 {ko(col)}', fontsize=12, fontweight='bold')
    ax.set_xlabel('CGPA 구간')
    ax.set_ylabel(ko(col))
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# 6-7. 성별 × CGPA × 위험도 (Facet 막대)
df_c = tdf(df, ['gender', 'cgpa_category', 'performance_risk_level'])
genders_ko = sorted(df_c['gender'].dropna().unique())

fig, axes = plt.subplots(1, len(genders_ko), figsize=(7 * len(genders_ko), 5), sharey=True)
if len(genders_ko) == 1:
    axes = [axes]

for ax, gender in zip(axes, genders_ko):
    sub = df_c[df_c['gender'] == gender]
    cnt = sub.groupby(['cgpa_category', 'performance_risk_level']).size().unstack(fill_value=0)
    cnt = cnt.reindex(cgpa_ko, fill_value=0)
    for r in risk_ko:
        if r not in cnt.columns:
            cnt[r] = 0
    cnt[risk_ko].plot(kind='bar', ax=ax,
                      color=[RISK_COLOR_KO[r] for r in risk_ko],
                      edgecolor='white', width=0.7)
    ax.set_title(f'성별: {gender}', fontsize=11, fontweight='bold')
    ax.set_xlabel('CGPA 구간')
    ax.tick_params(axis='x', rotation=20)
    ax.legend(title='성과 위험도', fontsize=8)

axes[0].set_ylabel('학생 수')
plt.suptitle('성별 × CGPA 구간 × 위험도', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 6-8. 학년 × 위험도별 평균 공부 시간 히트맵
study_num_map = {
    'Less than 1 hour': 0.5, '1\u20132 hours': 1.5, 'More than 2 hours': 3.0
}
df['study_hours_num'] = df['study_hours_daily'].map(study_num_map)
df_c = tdf(df, ['year_class', 'performance_risk_level'])
df_c['study_hours_num'] = df['study_hours_num']

pivot = df_c.pivot_table(
    values='study_hours_num', index='year_class',
    columns='performance_risk_level', aggfunc='mean'
)[risk_ko]

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlGnBu', linewidths=0.5, ax=ax)
ax.set_title('학년 × 위험도별 평균 공부 시간 (시간/일)', fontsize=12, fontweight='bold')
ax.set_xlabel('성과 위험도')
ax.set_ylabel('학년')
plt.tight_layout()
plt.show()

In [ ]:
# 6-9. 시험 준비 상태 × 위험도 비율 막대
df_c = tdf(df, ['preparation_status', 'performance_risk_level'])
prep_cross = pd.crosstab(df_c['preparation_status'], df_c['performance_risk_level'])
prep_cross = prep_cross.reindex(columns=risk_ko, fill_value=0)
prep_pct   = prep_cross.div(prep_cross.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
prep_pct.plot(kind='bar', stacked=True,
              color=[RISK_COLOR_KO[r] for r in risk_ko],
              ax=ax, edgecolor='white', width=0.65)
ax.set_title('시험 준비 상태별 위험도 비율', fontsize=12, fontweight='bold')
ax.set_xlabel('시험 준비 상태')
ax.set_ylabel('비율 (%)')
ax.tick_params(axis='x', rotation=20)
ax.legend(title='성과 위험도', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

---
## 7. EDA 인사이트 정리

| # | 발견 사항 | 분석 근거 |
|---|-----------|----------|
| 1 | **출석률 ↑ → 고CGPA 집중** | 교차 히트맵(6-1): `85% 초과` 구간에서 `8.5~9.4` 비율 최고 |
| 2 | **공부 시간 ↑ → 고CGPA 비율 증가** | 누적 막대(6-2): `2시간 초과` 그룹에서 고CGPA 분포 우편향 |
| 3 | **스트레스 ↑ · 에너지 ↓ → 고위험 경향** | 산점도(6-3): 고위험 클러스터가 고스트레스·저에너지 영역에 밀집 |
| 4 | **수면 6~7시간 그룹이 저위험 최다** | 카운트플롯(6-5): 과수면(8시간 초과)은 고위험 비율 증가 |
| 5 | **일일 생산성 ↑ → 고CGPA 뚜렷** | 바이올린(6-6): 생산성은 CGPA 구간에 따라 단조 증가 |
| 6 | **스트레스와 CGPA는 비선형** | 바이올린(6-6): 고CGPA 구간에서도 스트레스 분포 넓음 |
| 7 | **일찍 준비할수록 저위험 비율 높음** | 누적 막대(6-9): `적극 준비 중` 그룹이 저위험 최대 비율 |

### 모델링 방향

| 피처 | 중요도 예상 | 비고 |
|------|------------|------|
| 출석률 | ★★★★★ | CGPA 구간 분리 신호 최강 |
| 일일 공부 시간 | ★★★★ | 공부량 대리 변수 |
| 일일 생산성·에너지 | ★★★★ | 수치형, 직접 활용 가능 |
| 스트레스·수면 | ★★★ | 위험도 분류에 중요 |
| 준비 상태·복습 빈도 | ★★★ | 학습 습관 변수 |